In [1]:
import pandas as pd
import numpy as np

# --- Load data ---
df = pd.read_csv("Nat_Gas.csv")
df["Dates"] = pd.to_datetime(df["Dates"], format="%m/%d/%y")
df = df.sort_values("Dates").reset_index(drop=True)

# --- Convert dates to days since the first date ---
start = df["Dates"].min()
df["days"] = (df["Dates"] - start).dt.days
days = df["days"].values
prices = df["Prices"].values

# --- Fit the trend (straight line) ---
slope, intercept = np.polyfit(days, prices, 1)
trend = slope * days + intercept

# --- Fit the seasonality (yearly sine wave) ---
period = 365.25
theta = 2 * np.pi * days / period
residual = prices - trend
A, B = np.linalg.lstsq(np.column_stack([np.sin(theta), np.cos(theta)]), residual, rcond=None)[0]

# --- Estimate price for any date, past or ~1 year future ---
def estimate_price(date):
    date = pd.to_datetime(date)
    d = (date - start).days
    trend_part = slope * d + intercept
    theta = 2 * np.pi * d / period
    seasonal_part = A * np.sin(theta) + B * np.cos(theta)
    return round(trend_part + seasonal_part, 2)

print(estimate_price("2024-12-31"))
print(estimate_price("2025-06-30"))

12.8
11.93


In [2]:
def price_storage_contract(injection_dates, injection_prices,
                           withdrawal_dates, withdrawal_prices,
                           rate, max_volume, storage_cost_per_month):

    # 1. Build one chronological list of events, each carrying its price
    events = []
    for d, p in zip(injection_dates, injection_prices):
        events.append((pd.to_datetime(d), "inject", p))
    for d, p in zip(withdrawal_dates, withdrawal_prices):
        events.append((pd.to_datetime(d), "withdraw", p))
    events.sort(key=lambda e: e[0])          # process in date order

    # 2. Walk through time, tracking storage and cash
    volume = 0.0
    buy_cost = 0.0
    sell_revenue = 0.0
    for date, kind, price in events:
        if kind == "inject":
            amount = min(rate, max_volume - volume)   # can't exceed capacity
            volume += amount
            buy_cost += amount * price
        else:                                          # withdraw
            amount = min(rate, volume)                 # can't sell what you don't have
            volume -= amount
            sell_revenue += amount * price

    # 3. Storage cost over the life of the contract
    all_dates = [e[0] for e in events]
    months = ((max(all_dates).year - min(all_dates).year) * 12
              + (max(all_dates).month - min(all_dates).month))
    storage_cost = months * storage_cost_per_month

    # 4. Value = money in − money out
    return round(sell_revenue - buy_cost - storage_cost, 2)

In [3]:
inj_dates  = ["2024-07-30"]
inj_prices = [estimate_price("2024-07-30")]
wd_dates   = ["2024-12-31"]
wd_prices  = [estimate_price("2024-12-31")]

value = price_storage_contract(
    inj_dates, inj_prices,
    wd_dates, wd_prices,
    rate=1_000_000,             # up to 1M MMBtu moved per date
    max_volume=2_000_000,       # storage capacity
    storage_cost_per_month=100_000
)
print("Contract value:", value)

Contract value: 920000.0
